In [1]:
import pandas as pd

import numpy as np

from pathlib import Path
from scipy import stats
from scipy.stats import chi2

path = Path("results")

models = ["ds", "olmo", "phi", "qwen"]

q2_results = []

def mcnemar(col_a, col_b):
    """
    col_a, col_b: arrays of binary judgments on the same items.
    Uses the exact mid-p correction (good for large n; reduces to standard
    asymptotic chi² for n=1000 anyway).
    Returns: chi2 statistic, p-value
    """
    b = ((col_a == 1) & (col_b == 0)).sum()   # a correct, b wrong
    c = ((col_a == 0) & (col_b == 1)).sum()   # a wrong,  b correct
    print(f"b={b}, c={c}")
    if b + c == 0:
        return 0.0, 1.0
    chi2_stat = (abs(b - c) - 1)**2 / (b + c)  # with continuity correction
    p = 1 - chi2.cdf(chi2_stat, df=1)
    return chi2_stat, p

def holm_bonferroni(p_values, alpha=0.05):
    """
    Returns adjusted p-values and a boolean reject array.
    """
    n      = len(p_values)
    order  = np.argsort(p_values)
    sorted_p = np.array(p_values)[order]
 
    adjusted = np.zeros(n)
    for i, p in enumerate(sorted_p):
        adjusted[order[i]] = min(1.0, p * (n - i))
 
    # make monotone (each adjusted p >= previous)
    for i in range(1, n):
        adjusted[order[i]] = max(adjusted[order[i]], adjusted[order[i-1]])
 
    reject = adjusted < alpha
    return adjusted, reject

for eval_model in models:
    self_judgements = pd.read_json(path / f"{eval_model}_on_{eval_model}_full.jsonl", lines=True)["evaluator_judgment"].fillna(0)

    self_judgements = self_judgements.astype(int).to_numpy()

    for student_model in models: 
        if student_model == eval_model:
            continue
        
        judgements = pd.read_json(path / f"{eval_model}_on_{student_model}_full.jsonl", lines=True)["evaluator_judgment"].fillna(0)

        judgements = judgements.astype(int).to_numpy()

        chi2_stat, pval = mcnemar(self_judgements, judgements)
        q2_results.append({
            "Evaluator":       eval_model,
            "Evaluated model": student_model,
            "chi2":            round(chi2_stat, 3),
            "p-value":         pval,
        })

p_vals = [r["p-value"] for r in q2_results]
adj_p, reject = holm_bonferroni(p_vals)
for i, r in enumerate(q2_results):
    r["p-adj (Holm)"] = round(adj_p[i], 4)
    r["reject H0"]    = reject[i]
 
df_q2 = pd.DataFrame(q2_results)
df_q2["p-value"] = df_q2["p-value"].apply(lambda x: f"{x:.4f}")
print(f"\nMcNemar's: self-eval vs. each cross-evaluator (12 tests, Holm corrected):")
print(df_q2.to_string(index=False))




b=52, c=31
b=40, c=33
b=94, c=27
b=141, c=76
b=144, c=70
b=111, c=88
b=111, c=58
b=77, c=56
b=49, c=59
b=72, c=33
b=57, c=37
b=57, c=38

McNemar's: self-eval vs. each cross-evaluator (12 tests, Holm corrected):
Evaluator Evaluated model   chi2 p-value  p-adj (Holm)  reject H0
       ds            olmo  4.819  0.0281        0.1970      False
       ds             phi  0.493  0.4825        0.7730      False
       ds            qwen 36.000  0.0000        0.0000       True
     olmo              ds 18.876  0.0000        0.0001       True
     olmo             phi 24.902  0.0000        0.0000       True
     olmo            qwen  2.432  0.1189        0.3566      False
      phi              ds 16.000  0.0001        0.0006       True
      phi            olmo  3.008  0.0829        0.3315      False
      phi            qwen  0.750  0.3865        0.7730      False
     qwen              ds 13.752  0.0002        0.0017       True
     qwen            olmo  3.840  0.0500        0.3002      Fal

In [2]:
models = ["DeepSeek-R1-Distill-Qwen-32B",
"Olmo-3.1-32B-Think",
"phi-4",
"Qwen3-32B"]


povs = ["LLM", "she", "he", "they"]

In [3]:
q1_results = []

for model in models:
    data = pd.read_json(f"data/llm_verification/{model}_eval_reasoning_output.jsonl",  lines=True)

    print(f"\n{model}:")

    base_data = data[data["POV"] == "LLM"]["evaluator_judgment"].fillna(0).astype(int).to_numpy()

    for pov in povs:
        if pov == "LLM":
            continue
        
        pov_data = data[data["POV"] == pov]["evaluator_judgment"].fillna(0).astype(int).to_numpy()

        chi2_stat, pval = mcnemar(base_data, pov_data)
        print(f"  {pov}: chi2={chi2_stat:.3f}, p={pval:.4f}")

        q1_results.append({
            "Evaluator":       model,
            "Evaluated model": pov,
            "chi2":            round(chi2_stat, 3),
            "p-value":         pval,
        })

        

p_vals = [r["p-value"] for r in q1_results]
adj_p, reject = holm_bonferroni(p_vals)
for i, r in enumerate(q1_results):
    r["p-adj (Holm)"] = round(adj_p[i], 4)
    r["reject H0"]    = reject[i]

df_q1 = pd.DataFrame(q1_results)
df_q1["p-value"] = df_q1["p-value"].apply(lambda x: f"{x:.4f}")
print(f"\nMcNemar's: self-eval vs. each cross-evaluator (12 tests, Holm corrected):")
print(df_q1.to_string(index=False))


DeepSeek-R1-Distill-Qwen-32B:
b=36, c=27
  she: chi2=1.016, p=0.3135
b=37, c=24
  he: chi2=2.361, p=0.1244
b=30, c=24
  they: chi2=0.463, p=0.4962

Olmo-3.1-32B-Think:
b=52, c=39
  she: chi2=1.582, p=0.2084
b=46, c=38
  he: chi2=0.583, p=0.4450
b=45, c=42
  they: chi2=0.046, p=0.8302

phi-4:
b=15, c=26
  she: chi2=2.439, p=0.1183
b=14, c=28
  he: chi2=4.024, p=0.0449
b=8, c=29
  they: chi2=10.811, p=0.0010

Qwen3-32B:
b=24, c=30
  she: chi2=0.463, p=0.4962
b=35, c=26
  he: chi2=1.049, p=0.3057
b=32, c=27
  they: chi2=0.271, p=0.6025

McNemar's: self-eval vs. each cross-evaluator (12 tests, Holm corrected):
                   Evaluator Evaluated model   chi2 p-value  p-adj (Holm)  reject H0
DeepSeek-R1-Distill-Qwen-32B             she  1.016  0.3135        1.0000      False
DeepSeek-R1-Distill-Qwen-32B              he  2.361  0.1244        1.0000      False
DeepSeek-R1-Distill-Qwen-32B            they  0.463  0.4962        1.0000      False
          Olmo-3.1-32B-Think             she 